In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["figure.dpi"] = 110

rng = np.random.default_rng(seed=0)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def relu(z):
    return np.maximum(0.0, z)


def bce_loss(y_pred, y_true):
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


def plot_decision_boundary(predict_fn, X, y, title="", ax=None):
    """Shade the plane by what predict_fn says, then scatter the real data on top."""
    if ax is None:
        _, ax = plt.subplots()
    pad = 0.5
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = predict_fn(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=50, cmap="coolwarm", alpha=0.7)
    ax.contour(xx, yy, zz, levels=[0.5], colors="k", linewidths=2)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=35)
    ax.set_title(title)
    ax.set_xlabel("feature 1"); ax.set_ylabel("feature 2")
    return ax


print("ready")

---
# 4.1 A problem one neuron cannot solve

The classic example is **XOR**: four points, two classes. Output 1 when exactly one
input is 1, otherwise 0.

| x₁ | x₂ | label |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Look at the plot below and try to draw a single straight line with the two red points
on one side and the two blue points on the other. You can't. There is no such line —
this isn't a matter of not looking hard enough.

In [ ]:
X_xor = np.array([[0.0, 0.0],
                 [0.0, 1.0],
                 [1.0, 0.0],
                 [1.0, 1.0]])
y_xor = np.array([0.0, 1.0, 1.0, 0.0])

plt.figure(figsize=(4.5, 4.5))
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="coolwarm",
            edgecolor="k", s=300)
for (x1, x2), label in zip(X_xor, y_xor):
    plt.annotate(f"{int(label)}", (x1, x2), fontsize=13, fontweight="bold",
                 ha="center", va="center", color="white")
plt.xlim(-0.4, 1.4)
plt.ylim(-0.4, 1.4)
plt.show()

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=400, noise=0.20, random_state=3)
y = y.astype(float)

print("X", X.shape, " y", y.shape)

plt.figure(figsize=(5.5, 4.5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=30)
plt.show()

In [ ]:
def train_neuron(X, y, learning_rate=0.5, n_steps=500):
    n_samples, n_features = X.shape
    r = np.random.default_rng(0)
    w = r.normal(size=n_features) * 0.01
    b = 0.0
    for _ in range(n_steps):
        y_pred = sigmoid(X @ w + b)
        error = y_pred - y
        w -= learning_rate * (X.T @ error) / n_samples
        b -= learning_rate * np.mean(error)
    return w, b

w1n, b1n = train_neuron(X, y)
acc = np.mean((sigmoid(X @ w1n + b1n) > 0.5) == y)

plot_decision_boundary(lambda g: sigmoid(g @ w1n + b1n), X, y,
                       title = f"One neuron - accuracy {acc:.1%}")
plt.show()

In [ ]:
def init_params(n_input, n_hidden, seed=0):
    """Create the four parameter arrays."""
    r = np.random.default_rng(seed)
    return {
        "W1": r.normal(size=(n_input, n_hidden)) * np.sqrt(2.0 / n_input),
        "b1": np.zeros(n_hidden),
        "W2": r.normal(size=(n_hidden, 1)) * np.sqrt(2.0 / n_hidden),
        "b2": np.zeros(1),
    }

def forward(X, p):
    """Run the network."""
    Z1 = X @ p["W1"] + p["b1"]
    A1 = relu(Z1)
    Z2 = A1 @ p["W2"] + p["b2"]
    A2 = sigmoid(Z2)
    return A2, {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}

params = init_params(n_input=2, n_hidden=8)
for name, arr in params.items():
    print(f"{name}: shape {str(arr.shape):8}")

y_hat, cache = forward(X, params)
print()
print("predictions shape:", y_hat.shape)
print("first 3 (untrained, so near 0.5):", np.round(y_hat[:3, 0], 3))

In [ ]:
def backward(X, Y, p, cache):
    n = X.shape[0]
    A1, Z1, A2 = cache["A1"], cache["Z1"], cache["A2"]

    # ---- output layer ----
    dZ2 = (A2 - Y) / n
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(axis=0)

    # ---- pass the blame back to the hidden layer ----
    dA1 = dZ2 @ p["W2"].T
    dZ1 = dA1 * (Z1 > 0)

    # ---- hidden layer ----
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0)

    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

grads = backward(X, y.reshape(-1, 1), params, cache)
for name in params:
    print(f"{name}: param {str(params[name].shape):8} gradient {str(grads[name].shape):8}"
          f" {'MATCH' if params[name].shape == grads[name].shape else 'MISMATCH'}")

In [ ]:
def train_network(X, y, n_hidden=8, learning_rate=0.5, n_steps=3000, seed=0, snapshot_at=()):
    """Train the 2-layer network."""
    Y = y.reshape(-1, 1)  # (n, 1) - column form, always
    p = init_params(X.shape[1], n_hidden, seed=seed)

    losses = []
    snapshots = {}
    for step in range(n_steps):
        if step in snapshot_at:
            snapshots[step] = {k: v.copy() for k, v in p.items()}

        y_hat, cache = forward(X, p) # 1. forward
        losses.append(bce_loss(y_hat, Y))  # 2. loss
        grads = backward(X, Y, p, cache) # 3. backward

        for k in p:  # 4. update
            p[k] -= learning_rate * grads[k]

    if n_steps in snapshot_at:
        snapshots[n_steps] = {k: v.copy() for k, v in p.items()}
    return p, np.array(losses), snapshots

trained, losses, _ = train_network(X, y, n_hidden=8, learning_rate=0.5, n_steps=3000)

def predict_prob(grid, p):
    return forward(grid, p)[0].ravel()

acc = np.mean((predict_prob(X, trained) > 0.5) == y)
print(f"loss  {losses[0]:.4f}  ->  {losses[-1]:.4f}")
print(f"accuracy: {acc:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(losses, linewidth=2)
axes[0].set_xlabel("training step")
axes[0].set_ylabel("loss")
axes[0].set_title("Loss")

plot_decision_boundary(lambda g: predict_prob(g, trained), X, y,
                       title=f"2-layer network - {acc:.1%}", ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
steps = [0, 50, 200, 1000, 3000]
_, _, snaps = train_network(X, y, n_hidden=8, learning_rate=0.5,
                            n_steps=3000, snapshot_at=steps)
fig, axes = plt.subplots(1, len(steps), figsize=(4 * len(steps), 4))
for ax, s in zip(axes, steps):
    p_s = snaps[s]
    a = np.mean((predict_prob(X, p_s) > 0.5) == y)
    plot_decision_boundary(lambda g, pp=p_s: predict_prob(g, pp), X, y,
                           title=f"step {s} - {a:.0%}", ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
p_xor, losses_xor, _ = train_network(X_xor, y_xor, n_hidden=4,
                                     learning_rate=1.0, n_steps=5000, seed=1)

pred = predict_prob(X_xor, p_xor)
print("  x1   x2   target   predicted")
for (x1, x2), t, q in zip(X_xor, y_xor, pred):
    print(f"  {x1:.0f}    {x2:.0f}      {t:.0f}        {q:.3f}")

plot_decision_boundary(lambda g : predict_prob(g, p_xor), X_xor, y_xor,
                       title="XOR, solved")

## Experiment 1: remove the activation function

This is the important one. We claimed in notebook 1 that without a non-linear
activation, depth buys you nothing. Let's prove it by replacing ReLU with the
identity function — the hidden layer still exists, still has eight units, still
trains, but no longer bends anything.

In [ ]:
def forward_linear(X, p):
    Z1 = X @ p["W1"] + p["b1"]
    A1 = Z1
    Z2 = A1 @ p["W2"] + p["b2"]
    A2 = sigmoid(Z2)
    return A2, {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}

def backward_linear(X, Y, p, cache):
    n = X.shape[0]
    dZ2 = (cache["A2"] - Y) / n
    dW2 = cache["A1"].T @ dZ2
    db2 = dZ2.sum(axis=0)
    dA1 = dZ2 @ p["W2"].T
    dZ1 = dA1 
    return {"W1": X.T @ dZ1, "b1": dZ1.sum(axis=0), "W2": dW2, "b2": db2}

p_lin = init_params(2, 8, seed=0)  # create the network's weights
Y = y.reshape(-1, 1)
for _ in range(3000):
    yh, c = forward_linear(X, p_lin)  # 1. make predictions
    p_lin_grads = backward_linear(X, Y, p_lin, c)  # 2. compute gradients
    for k in p_lin:  # 3. update
        p_lin[k] -= 0.5 * p_lin_grads[k]

acc_lin = np.mean((forward_linear(X, p_lin)[0].ravel() > 0.5) == y)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_decision_boundary(lambda g: forward_linear(g, p_lin)[0].ravel(), X, y,
                       title=f"8 hidden units, NO activation - {acc_lin:.1%}", ax=axes[0])
plot_decision_boundary(lambda g: predict_prob(g, trained), X, y,
                       title=f"8 hidden units, WITH ReLU - {acc:.1%}", ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
sizes = [1, 2, 4, 8, 32, 128]
fig, axes = plt.subplots(1, len(sizes), figsize=(3.6 * len(sizes), 3.8))

for ax, h in zip(axes, sizes):
    p_h, _, _ = train_network(X, y, n_hidden=h, learning_rate=0.5, n_steps=4000)
    a = np.mean((predict_prob(X, p_h) > 0.5) == y)
    plot_decision_boundary(lambda g, pp=p_h: predict_prob(g, pp), X, y,
                           title=f"{h} hidden units - {a:.0%}", ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
print("accuracy by hidden size and random seed")
print(f"{'h':>4} | " + "  ".join(f"seed {s}" for s in range(6)))
print("-" * 52)
for h in [2, 4, 8, 32]:
    row = []
    for s in range(6):
        p_h, _, _, = train_network(X, y, n_hidden=h, learning_rate=0.5,
                                   n_steps=4000, seed=s)
        row.append(np.mean((predict_prob(X, p_h) > 0.5) == y))
    print(f"{h:>4} | " + " ".join(f"{v:5.1%} " for v in row))

In [ ]:
plt.figure(figsize=(7.5, 4.5))
for lr in [0.01, 0.1, 1.0, 5.0, 50.0]:
    _, l, _ = train_network(X, y, n_hidden=8, learning_rate=lr, n_steps=2000)
    plt.plot(l, label=f"lr = {lr}", linewidth=2)

plt.xlabel("training step")
plt.ylabel("loss")
plt.title("Learning rate on the 2-layer network")
plt.legend()
plt.show()